# Residue–residue interactions in the GXG condensates (ProLIF)

Two peptide lengths, five compositions each:

| | peptide | systems |
|---|---|---|
| `3_res` | `GXG` (3 residues/chain) | equal, aromatic, charged, nonpolar, polar |
| `9_res` | `GGGGXGGGG` (9 residues/chain) | equal (= `old_equal`), aromatic, charged, nonpolar, polar |

Each `<RES>_plf.csv` has ligand = every residue of that type, protein = every residue, one row per frame
(3-row header: ligand / protein / interaction). ProLIF resids are 0-based and sequential, so chain = resid // residues-per-chain
(checked against `output-nw.pdb` for every system).

**Counting rules** — each physical contact is counted once per residue pair:

| Category | ProLIF types used | Why |
|---|---|---|
| H-Bonding | `HBAcceptor` from both files | `HBDonor` in the partner file is the same bond seen from the donor side |
| Electrostatic | `Cationic` from both files | `Anionic` is its mirror |
| Pi-Cation | `PiCation` from both files | `CationPi` is its mirror |
| Pi-Stacking | `PiStacking`, averaged over both directions | `FaceToFace` / `EdgeToFace` are subtypes of `PiStacking` |
| Hydrophobic, VdW | averaged over both directions | symmetric |

The two directions are **not** detected identically for H-bonds / electrostatics / π-cation (see section 3),
so this is a choice; change `CATS` to use the other side.

Contacts within the same peptide (bonded neighbours) are dropped. Values are **contacts per X residue per frame**.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'gxg_comps'          # relative to this notebook (gxg-project/)
OUT = 'plf_interactions'
CACHE_DIR = f'{OUT}/cache'
PLOT_DIR = f'{OUT}/plots'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

COMPS = ['equal', 'aromatic', 'charged', 'nonpolar', 'polar']
LENGTHS = {'3_res': 3, '9_res': 9}                     # residues per chain
DIRS = {('3_res', c): f'{BASE}/3_res/{c}' for c in COMPS}
DIRS.update({('9_res', c): f'{BASE}/9_res/{c}' for c in COMPS})
DIRS[('9_res', 'equal')] = f'{BASE}/9_res/old_equal'   # 9_res/equal has no per-residue plf files

In [ ]:
DIRS

## 1. Reduce the fingerprints
Sums every (ligand, protein, interaction) column over frames, one cache file per system.
Reading the ~200 CSVs over the network takes a while the first time; delete a cache file to recompute that system.

In [ ]:
def reduce_system(length, comp):
    rows = []
    per_chain = LENGTHS[length]
    for fn in sorted(glob.glob(f'{DIRS[length, comp]}/plf_files/*_plf.csv')):
        df = pd.read_csv(fn, header=[0, 1, 2], index_col=0, low_memory=False)
        n_frames = len(df)
        for (lig, prot, inter), c in df.sum(axis=0).items():
            lig_i, prot_i = int(lig[3:]), int(prot[3:])
            rows.append((length, comp, lig[:3], lig_i, prot[:3], prot_i, inter, int(c), n_frames,
                         lig_i // per_chain == prot_i // per_chain))
        print(length, comp, os.path.basename(fn), n_frames)
    return pd.DataFrame(rows, columns=['length', 'comp', 'lig_res', 'lig_id', 'prot_res', 'prot_id',
                                       'interaction', 'count', 'n_frames', 'same_chain'])

parts = []
for length, comp in DIRS:
    cache = f'{CACHE_DIR}/{length}_{comp}.csv'
    if not os.path.exists(cache):
        reduce_system(length, comp).to_csv(cache, index=False)
    parts.append(pd.read_csv(cache))
raw = pd.concat(parts, ignore_index=True)
raw.groupby(['length', 'comp']).n_frames.agg(['min', 'max', 'size'])

## 2. Residue counts per system (from `output-nw.pdb`)

In [ ]:
def res_counts(path):
    names, prev = [], None
    for line in open(f'{path}/output-nw.pdb'):
        if line.startswith('ATOM'):
            key = (line[21], line[22:26])
            if key != prev:
                names.append(line[17:20])
                prev = key
    return pd.Series(names).value_counts()

nres = {key: res_counts(path) for key, path in DIRS.items()}
pd.DataFrame(nres).fillna(0).astype(int)

## 3. Sanity check: mirrored interaction types
For every residue pair, `HBAcceptor` A→B should match `HBDonor` B→A, `Cationic` ↔ `Anionic`, `PiCation` ↔ `CationPi`,
and `PiStacking` should match itself. Totals over each system:

In [ ]:
IDX = ['length', 'comp', 'lig_res', 'prot_res']
inter = raw[~raw.same_chain].groupby(IDX + ['interaction'])['count'].sum().unstack(fill_value=0)

def mirror(a, b):
    x = inter[a] if a in inter else pd.Series(0, index=inter.index)
    y = (inter[b] if b in inter else pd.Series(0, index=inter.index)) \
        .rename_axis(['length', 'comp', 'prot_res', 'lig_res']).reorder_levels(IDX)
    return pd.concat([x.rename(a), y.rename(f'{b} (mirrored)')], axis=1).fillna(0) \
        .groupby(level=['length', 'comp']).sum().astype(int)

for a, b in [('HBAcceptor', 'HBDonor'), ('Cationic', 'Anionic'), ('PiCation', 'CationPi'), ('PiStacking', 'PiStacking')]:
    print(f'--- {a} vs {b}'); print(mirror(a, b), '\n')

## 4. Pair table and plotting helpers

In [ ]:
CATS = {  # bar category -> (ProLIF types, symmetric?)
    'Pi-Stacking':   (['PiStacking'], True),
    'H-Bonding':     (['HBAcceptor'], False),
    'Pi-Cation':     (['PiCation'], False),
    'Electrostatic': (['Cationic'], False),
    'Hydrophobic':   (['Hydrophobic'], True),
    'VdW':           (['VdWContact'], True),
}
BAR_CATS = ['Pi-Stacking', 'H-Bonding', 'Pi-Cation', 'Electrostatic']
CAT_COLOR = {'Pi-Stacking': '#e6d600', 'H-Bonding': '#86bc45', 'Pi-Cation': '#e5004c',
             'Electrostatic': '#0d8be6', 'Hydrophobic': '#c27fc4', 'VdW': '#888888'}
CLASS = {**dict.fromkeys(['ALA', 'GLY', 'VAL', 'PRO', 'ILE', 'LEU', 'MET'], 'Nonpolar'),
         **dict.fromkeys(['CYS', 'THR', 'SER', 'GLN', 'ASN'], 'Polar'),
         **dict.fromkeys(['PHE', 'TYR', 'TRP'], 'Aromatic'),
         **dict.fromkeys(['ASP', 'GLU'], 'Negative'),
         **dict.fromkeys(['LYS', 'ARG', 'HIS'], 'Positive')}
CLASS_COLOR = {'Positive': '#e5004c', 'Aromatic': '#d4c400', 'Polar': '#86bc45',
               'Negative': '#0d8be6', 'Nonpolar': '#c27fc4'}
MARKERS = dict(zip(COMPS, ['o', 's', 'D', '^', 'v']))


def pair_table(df):
    """(length, comp, res, partner) -> contacts/frame summed over residue pairs, both directions folded in."""
    df = df[~df.same_chain].copy()
    df['rate'] = df['count'] / df['n_frames']
    out = []
    for cat, (types, sym) in CATS.items():
        g = df[df.interaction.isin(types)].groupby(IDX)['rate'].sum()
        g = g.add(g.rename_axis(['length', 'comp', 'prot_res', 'lig_res']).reorder_levels(IDX), fill_value=0)
        if sym:
            g = g / 2
        out.append(g.rename(cat))
    pt = pd.concat(out, axis=1).fillna(0)
    pt.index.names = ['length', 'comp', 'res', 'partner']
    return pt


def tick_colors(labels, ticks):
    for t, r in zip(ticks, labels):
        t.set_color(CLASS_COLOR[CLASS[r]])


def stacked_panel(ax, data, title, ylabel):
    data = data.loc[data[BAR_CATS].sum(axis=1).sort_values().index]
    x, bottom = np.arange(len(data)), np.zeros(len(data))
    for cat in BAR_CATS:
        ax.bar(x, data[cat], 0.55, bottom=bottom, color=CAT_COLOR[cat], label=cat)
        bottom += data[cat].values
    ax.set_xticks(x, data.index, rotation=45, ha='right', rotation_mode='anchor')
    tick_colors(data.index, ax.get_xticklabels())
    ax.set_title(title, loc='left', fontsize=13)
    ax.set_ylabel(ylabel)
    ax.set_xlim(-0.6, len(data) - 0.4)


def dot_panel(ax, per_comp, cat, ylabel, order):
    x = np.arange(len(order))
    for comp, d in per_comp.items():
        v = d.reindex(order)[cat]
        ax.scatter(x, v, marker=MARKERS[comp], s=40, c=[CLASS_COLOR[CLASS[r]] for r in order],
                   edgecolors='k', linewidths=0.4, label=comp, zorder=3)
    ax.set_xticks(x, order, rotation=45, ha='right', rotation_mode='anchor')
    tick_colors(order, ax.get_xticklabels())
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)
    leg = ax.legend(title='Composition', fontsize=9, title_fontsize=9)
    for h in leg.legend_handles:
        h.set_facecolor('w')


def make_figure(per_comp, dot_cat, ylabel, fname, suptitle):
    """Panels A-E: stacked bars per composition (shared y); F: dot_cat across compositions."""
    fig = plt.figure(figsize=(14, 18))
    gs = fig.add_gridspec(4, 2)
    bar_axes = []
    for i, comp in enumerate(per_comp):
        ax = fig.add_subplot(gs[i // 2, i % 2])
        stacked_panel(ax, per_comp[comp], f'{"ABCDE"[i]}   {comp}', ylabel)
        bar_axes.append(ax)
    ymax = max(a.get_ylim()[1] for a in bar_axes)
    for a in bar_axes:
        a.set_ylim(0, ymax)
    leg_ax = fig.add_subplot(gs[2, 1])
    leg_ax.axis('off')
    leg_ax.legend(*bar_axes[0].get_legend_handles_labels(), loc='center', fontsize=13, title='Interaction',
                  title_fontsize=13)
    order = pd.concat(per_comp).groupby(level=1)[dot_cat].mean().sort_values().index
    ax = fig.add_subplot(gs[3, :])
    dot_panel(ax, per_comp, dot_cat, f'# of {dot_cat} contacts', order)
    ax.set_title(f'F   {dot_cat}', loc='left', fontsize=13)
    fig.suptitle(suptitle, fontsize=15)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{PLOT_DIR}/{fname}.{ext}', dpi=300)
    plt.show()


pt = pair_table(raw)
pt.head()

## 5. Glycine interactions with each amino acid
Gly–X contacts per **X residue** per frame (Gly–Gly left out). Panels A–E: specific interactions per composition;
panel F: VdW contacts across compositions (Gly has no side chain, so VdW is most of its contact).

In [ ]:
def gly_x(length):
    per_comp = {}
    for comp in COMPS:
        d = pt.loc[(length, comp, 'GLY')].drop(index='GLY', errors='ignore')
        per_comp[comp] = d.div(nres[length, comp].reindex(d.index), axis=0)
    return per_comp

gly = {L: gly_x(L) for L in LENGTHS}
for L in LENGTHS:
    make_figure(gly[L], 'VdW', '# of contacts with Gly per residue',
                f'{L}_gly_X_interactions', f'{L}: glycine interactions with each amino acid')
pd.concat({L: pd.concat(gly[L]) for L in LENGTHS}, names=['length', 'comp', 'partner']) \
    .to_csv(f'{PLOT_DIR}/gly_X_interactions.csv')

## 6. Amino acid interactions with non-glycine residues
For each X ≠ Gly: contacts with all non-Gly partners (including other X), per X residue per frame.
Panel F: hydrophobic contacts across compositions.

In [ ]:
def x_nongly(length):
    per_comp = {}
    for comp in COMPS:
        d = pt.loc[(length, comp)]
        keep = (d.index.get_level_values('res') != 'GLY') & (d.index.get_level_values('partner') != 'GLY')
        d = d[keep].groupby(level='res').sum()
        per_comp[comp] = d.div(nres[length, comp].reindex(d.index), axis=0)
    return per_comp

nongly = {L: x_nongly(L) for L in LENGTHS}
for L in LENGTHS:
    make_figure(nongly[L], 'Hydrophobic', '# of contacts with non-Gly residues per residue',
                f'{L}_X_nonGly_interactions', f'{L}: amino acid interactions with non-glycine residues')
pd.concat({L: pd.concat(nongly[L]) for L in LENGTHS}, names=['length', 'comp', 'res']) \
    .to_csv(f'{PLOT_DIR}/X_nonGly_interactions.csv')

## 7. Residue × residue map (Gly excluded)
Specific contacts (H-bond + electrostatic + π + hydrophobic) between each pair of types, per **row** residue per frame.

In [ ]:
for L in LENGTHS:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    for ax, comp in zip(axes.flat, COMPS):
        d = pt.loc[(L, comp)][BAR_CATS + ['Hydrophobic']].sum(axis=1).unstack(fill_value=0) \
              .drop(index='GLY', columns='GLY')
        d = d.div(nres[L, comp].reindex(d.index), axis=0)
        im = ax.imshow(d.values, cmap='viridis')
        ax.set_xticks(range(len(d.columns)), d.columns, rotation=90)
        ax.set_yticks(range(len(d.index)), d.index)
        tick_colors(d.columns, ax.get_xticklabels())
        tick_colors(d.index, ax.get_yticklabels())
        ax.set_title(comp, loc='left')
        fig.colorbar(im, ax=ax, shrink=0.8, label='contacts per row residue per frame')
    axes.flat[-1].axis('off')
    fig.suptitle(f'{L}: residue–residue specific contacts, Gly excluded')
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{PLOT_DIR}/{L}_X_Y_heatmap.{ext}', dpi=300)
    plt.show()

In [ ]:
from matplotlib.colors import Normalize, LinearSegmentedColormap

# ---- pick ONE colormap (single-hue, white -> dark) ----
CMAP = 'Blues'
# Other single-hue options: 'Purples', 'Reds', 'Oranges', 'Greys'
# Two-tone but smooth:      'PuBu', 'BuPu', 'RdPu', 'PuRd', 'OrRd'
# Custom (white -> any color you like):
# CMAP = LinearSegmentedColormap.from_list('custom', ['#ffffff', '#08306b'])   # navy
# CMAP = LinearSegmentedColormap.from_list('custom', ['#ffffff', '#67000d'])   # dark red
# CMAP = LinearSegmentedColormap.from_list('custom', ['#ffffff', '#3f007d'])   # deep purple
# CMAP = LinearSegmentedColormap.from_list('custom', ['#ffffff', '#00441b'])   # teal-green

def contact_matrix(L, comp):
    d = pt.loc[(L, comp)][BAR_CATS + ['Hydrophobic']].sum(axis=1).unstack(fill_value=0) \
          .drop(index='GLY', columns='GLY')
    return d.div(nres[L, comp].reindex(d.index), axis=0)

# ---- one common scale for ALL lengths and compositions ----
mats = {(L, comp): contact_matrix(L, comp) for L in LENGTHS for comp in COMPS}
vmax = max(np.nanmax(m.values) for m in mats.values())
norm = Normalize(vmin=0, vmax=vmax)

for L in LENGTHS:
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    for ax, comp in zip(axes.flat, COMPS):
        d = mats[(L, comp)]
        im = ax.imshow(d.values, cmap=CMAP, norm=norm)
        ax.set_xticks(range(len(d.columns)), d.columns, rotation=90)
        ax.set_yticks(range(len(d.index)), d.index)
        tick_colors(d.columns, ax.get_xticklabels())
        tick_colors(d.index, ax.get_yticklabels())
        ax.set_title(comp, loc='left')
    axes.flat[-1].axis('off')

    fig.suptitle(f'{L}: residue–residue specific contacts, Gly excluded')
    fig.tight_layout()
    # one shared colorbar for the whole figure
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6,
                 label='contacts per row residue per frame')
    for ext in ('png', 'pdf'):
        fig.savefig(f'{PLOT_DIR}/{L}_X_Y_heatmap.{ext}', dpi=300, bbox_inches='tight')
    plt.show()